In [1]:
#!/usr/bin/env python3
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.6 - CLASIFICACIÓN (GRÁFICAS)
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# DESCRIPCIÓN: Genera gráficas a partir de los resultados guardados
#              en el Paso 5.2 (entrenamiento).
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix   # <--- NUEVA IMPORTACIÓN
import warnings
warnings.filterwarnings('ignore')

# Configurar matplotlib para guardar sin ventana
import matplotlib
matplotlib.use('Agg')

print("="*60)
print("STEP 6: GENERATING CLASSIFICATION PLOTS")
print("="*60)

INPUT_DIR = 'ml_results_classification'
OUTPUT_DIR = 'ml_results_classification'

# ==========================================
# 1. CARGAR RESULTADOS
# ==========================================
with open(os.path.join(INPUT_DIR, 'results.pkl'), 'rb') as f:
    results = pickle.load(f)
with open(os.path.join(INPUT_DIR, 'predictions.pkl'), 'rb') as f:
    predictions = pickle.load(f)
y_test = np.load(os.path.join(INPUT_DIR, 'y_test.npy'))
cm_best = np.load(os.path.join(INPUT_DIR, 'confusion_matrix.npy'))  # Matriz del mejor modelo
df_summary = pd.read_excel(os.path.join(INPUT_DIR, 'models_summary.xlsx'))
df_summary = df_summary.set_index(df_summary.columns[0])

# Cargar importancia de características (si existe)
try:
    df_imp = pd.read_excel(os.path.join(INPUT_DIR, 'feature_importance.xlsx'))
    importances_available = True
except:
    importances_available = False

# ===== CAMBIO: 3 categorías =====
class_names = ['Vacía (0)', 'Baja (1-20)', 'Alta (>20)']
best_model = df_summary.index[0]
top_models = df_summary.index[:3].tolist()

print(f"   Mejor modelo: {best_model}")
print(f"   Top 3 modelos: {top_models}")
print(f"   Modelos cargados: {len(df_summary)}")

# ==========================================
# 2. GRÁFICA 1: Comparativa de métricas (barras agrupadas)
# ==========================================
fig, ax = plt.subplots(figsize=(12, 6))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'CV_F1_mean']
x = np.arange(len(df_summary))
width = 0.15
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    values = df_summary[metric].values if metric in df_summary.columns else [0]*len(df_summary)
    offset = (i - len(metrics)/2) * width + width/2
    bars = ax.bar(x + offset, values, width, label=metric, color=color, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_xlabel('Models', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Comparativa de métricas (todos los modelos)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_summary.index, fontsize=9, rotation=15, ha='right')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'metrics_comparison.png'), dpi=150)
plt.close()
print("   ✅ metrics_comparison.png")

# ==========================================
# 3. GRÁFICA 2: Matriz de confusión del MEJOR modelo (individual)
# ==========================================
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={"size": 12})
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax.set_title(f'Matriz de confusión - {best_model}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix_best.png'), dpi=150)
plt.close()
print("   ✅ confusion_matrix_best.png")

# ==========================================
# 4. GRÁFICA 3: Matrices de confusión para los 3 MEJORES modelos
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Matrices de Confusión - Top 3 Modelos', fontsize=14, fontweight='bold')

for idx, model_name in enumerate(top_models):
    ax = axes[idx]
    y_pred = predictions[model_name]
    cm_top = confusion_matrix(y_test, y_pred, labels=range(3))   # <--- 3 clases
    sns.heatmap(cm_top, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 12}, cbar=False)
    ax.set_xlabel('Predicted', fontsize=11, fontweight='bold')
    ax.set_ylabel('Actual', fontsize=11, fontweight='bold')
    ax.set_title(f'{model_name}\nF1: {df_summary.loc[model_name, "F1"]:.3f}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix_top3.png'), dpi=150)
plt.close()
print("   ✅ confusion_matrix_top3.png")

# ==========================================
# 5. GRÁFICA 4: Importancia de características (si existe)
# ==========================================
if importances_available and len(df_imp) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    top_n = min(15, len(df_imp))
    top_features = df_imp.head(top_n)
    ax.barh(range(top_n), top_features['Importance'][::-1], color='steelblue', edgecolor='navy')
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_features['Feature'][::-1], fontsize=9)
    ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
    ax.set_title('Importancia de características (Random Forest)', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=150)
    plt.close()
    print("   ✅ feature_importance.png")
else:
    print("   ⚠️ Importancia de características no disponible")

# ==========================================
# 6. GRÁFICA 5: Distribución de categorías en train y test (3 categorías)
# ==========================================
INPUT_NORM = 'ml_normalized_grouped'
y_train_orig = pd.read_excel(os.path.join(INPUT_NORM, 'y_train.xlsx')).values.ravel()
y_test_orig  = pd.read_excel(os.path.join(INPUT_NORM, 'y_test.xlsx')).values.ravel()

def categorize(val):
    if val == 0:    return 'Vacía (0)'
    elif val <= 20: return 'Baja (1-20)'
    else:           return 'Alta (>20)'

cat_order = ['Vacía (0)', 'Baja (1-20)', 'Alta (>20)']
colors_cat = ['#bdc3c7', '#2ecc71', '#e74c3c']

train_cat = pd.Series(y_train_orig).apply(categorize)
test_cat  = pd.Series(y_test_orig).apply(categorize)

counts_train = train_cat.value_counts().reindex(cat_order).fillna(0)
counts_test  = test_cat.value_counts().reindex(cat_order).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Distribución de categorías en entrenamiento y test', fontsize=14, fontweight='bold')

for ax, (data, title) in zip(axes, [(counts_train, 'Entrenamiento'), (counts_test, 'Test')]):
    bars = ax.bar(cat_order, data.values, color=colors_cat, edgecolor='white', linewidth=0.5)
    ax.set_ylabel('Número de bloques', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{int(val)}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution_train_test.png'), dpi=150)
plt.close()
print("   ✅ class_distribution_train_test.png")

# ==========================================
# 7. GRÁFICA 6: Train vs Validation Accuracy (diagnóstico)
# ==========================================
if 'Train_Acc' in df_summary.columns and 'Val_Acc' in df_summary.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    models = df_summary.index.tolist()
    x = np.arange(len(models))
    width = 0.35

    train_vals = df_summary['Train_Acc'].values
    val_vals   = df_summary['Val_Acc'].values

    bars1 = ax.bar(x - width/2, train_vals, width, label='Train Accuracy', color='#3498db', edgecolor='black')
    bars2 = ax.bar(x + width/2, val_vals, width, label='Validation Accuracy', color='#e67e22', edgecolor='black')

    if 'Diagnosis' in df_summary.columns:
        diag_colors = {'✅ BUEN AJUSTE': 'green', '🟡 OVERFITTING LEVE': 'orange',
                       '⚠️ OVERFITTING': 'red', '🚨 OVERFITTING SEVERO': 'darkred',
                       '🔴 UNDERFITTING': 'purple', '🔴 UNDERFITTING SEVERO': 'maroon'}
        for i, model in enumerate(models):
            diag = df_summary.loc[model, 'Diagnosis']
            color = diag_colors.get(diag, 'black')
            ax.text(x[i], max(train_vals[i], val_vals[i]) + 0.02, diag,
                    ha='center', fontsize=8, color=color, weight='bold')

    ax.set_xlabel('Models', fontsize=12, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax.set_title('Train vs Validation Accuracy con diagnóstico', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=15, ha='right', fontsize=9)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 1.05)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'train_val_accuracy_gap.png'), dpi=150)
    plt.close()
    print("   ✅ train_val_accuracy_gap.png")
else:
    print("   ⚠️ No se encontraron Train_Acc / Val_Acc para el gráfico de gap")

# ==========================================
# 8. GRÁFICA 7: Heatmap de correlación de métricas
# ==========================================
if 'Accuracy' in df_summary.columns:
    df_metrics = df_summary[['Accuracy', 'Precision', 'Recall', 'F1', 'CV_F1_mean']].dropna()
    if len(df_metrics) > 1:
        corr = df_metrics.corr()
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', square=True, ax=ax, annot_kws={"size": 10})
        ax.set_title('Correlación entre métricas', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'metrics_correlation.png'), dpi=150)
        plt.close()
        print("   ✅ metrics_correlation.png")
    else:
        print("   ⚠️ No hay suficientes datos para correlación")

print("\n" + "="*60)
print("✅ GRÁFICAS GENERADAS")
print("="*60)
print(f"   Archivos guardados en: {OUTPUT_DIR}/")
print("   Lista de gráficas:")
print("   - metrics_comparison.png")
print("   - confusion_matrix_best.png")
print("   - confusion_matrix_top3.png")
if importances_available:
    print("   - feature_importance.png")
print("   - class_distribution_train_test.png")
print("   - train_val_accuracy_gap.png")
print("   - metrics_correlation.png (si aplica)")
print("="*60)

STEP 6: GENERATING CLASSIFICATION PLOTS
   Mejor modelo: Gradient Boosting
   Top 3 modelos: ['Gradient Boosting', 'Decision Tree', 'Random Forest']
   Modelos cargados: 5
   ✅ metrics_comparison.png
   ✅ confusion_matrix_best.png
   ✅ confusion_matrix_top3.png
   ✅ feature_importance.png
   ✅ class_distribution_train_test.png
   ✅ train_val_accuracy_gap.png
   ✅ metrics_correlation.png

✅ GRÁFICAS GENERADAS
   Archivos guardados en: ml_results_classification/
   Lista de gráficas:
   - metrics_comparison.png
   - confusion_matrix_best.png
   - confusion_matrix_top3.png
   - feature_importance.png
   - class_distribution_train_test.png
   - train_val_accuracy_gap.png
   - metrics_correlation.png (si aplica)
